# Movie Recommender System with Word2Vec - T01 Reza Naserivand

---

**Core idea:** Movies that a user watches together (like songs in a playlist) appear side by side.  
Word2Vec learns from this co-watching behavior which movies are close to each other in embedding space.

---

## Dataset: MovieLens 20M

| Attribute | Value |
|-----------|-------|
| Source | [Kaggle — grouplens/movielens-20m-dataset](https://www.kaggle.com/datasets/grouplens/movielens-20m-dataset) |
| Ratings | 20,000,263 |
| Movies | 27,278 |
| Users | 138,493 |
| Period | Jan 1995 – Mar 2015 |
| License | GroupLens Research — free for non-commercial use |

**Files used:**
- `rating.csv` — `userId, movieId, rating, timestamp`
- `movie.csv` — `movieId, title, genres`

**Playlist logic:**  
Each **user** = one playlist ← movies they rated are the Word2Vec "sentence"

---
## 1. Load Dataset

In [ ]:
import pandas as pd
import numpy as np

RATINGS_PATH = '/kaggle/input/datasets/organizations/grouplens/movielens-20m-dataset/rating.csv'
MOVIES_PATH  = '/kaggle/input/datasets/organizations/grouplens/movielens-20m-dataset/movie.csv'

print('Loading ratings...')
ratings = pd.read_csv(RATINGS_PATH)

print('Loading movies...')
movies = pd.read_csv(MOVIES_PATH)

print(f'\n ratings : {ratings.shape[0]:,} rows  |  {ratings["userId"].nunique():,} users  |  {ratings["movieId"].nunique():,} movies')
print(f'movies  : {movies.shape[0]:,} movies with metadata')
ratings.head()

Loading ratings...
Loading movies...

 ratings : 20,000,263 rows  |  138,493 users  |  26,744 movies
movies  : 27,278 movies with metadata


,userId,movieId,rating,timestamp
0,1,2,3.5,2005-04-02 23:53:47
1,1,29,3.5,2005-04-02 23:31:16
2,1,32,3.5,2005-04-02 23:33:39
3,1,47,3.5,2005-04-02 23:32:07
4,1,50,3.5,2005-04-02 23:29:40


In [47]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


---
## 2. Build Movie "Playlists"

Each user = one playlist ← only movies rated highly (≥ 4.0)

In [50]:
# Keep only movies the user genuinely liked (rating 4.0 or above)
liked = ratings[ratings['rating'] >= 4.0].copy()

# Each user = one playlist of movieIds they liked
print('Building playlists...')
playlists = (
    liked.groupby('userId')['movieId']
    .apply(lambda x: list(x.astype(str)))
    .tolist()
)

# Drop playlists that are too short (fewer than 10 movies)
playlists = [p for p in playlists if len(p) >= 10]

# Lookup table: movieId → title + genres
movie_meta = movies.set_index('movieId')[['title', 'genres']].copy()
movie_meta.index = movie_meta.index.astype(str)

print(f'Playlists (users with ≥10 liked movies) : {len(playlists):,}')
print(f'Average playlist length: {np.mean([len(p) for p in playlists]):.1f} movies')
print(f'\nSample playlist (movie IDs):\n{playlists[0]}')

Building playlists...
Playlists (users with ≥10 liked movies) : 129,797
Average playlist length: 76.6 movies

Sample playlist (movie IDs):
['151', '223', '253', '260', '293', '296', '318', '541', '1036', '1079', '1090', '1097', '1196', '1198', '1200', '1214', '1215', '1219', '1240', '1249', '1258', '1259', '1266', '1278', '1321', '1333', '1358', '1374', '1387', '1967', '2021', '2100', '2118', '2138', '2140', '2143', '2173', '2174', '2193', '2288', '2291', '2542', '2628', '2762', '2872', '2944', '2959', '2968', '3081', '3153', '3479', '3489', '3499', '3889', '3996', '4011', '4027', '4128', '4306', '4467', '4571', '4754', '4896', '4911', '4993', '5026', '5039', '5171', '5540', '5797', '5816', '5952', '6093', '6333', '6539', '6754', '6774', '7046', '7153', '7389', '7438', '7454', '7757', '8368', '8507', '8636', '8961', '31696']


---
## 3. Train Word2Vec Model

In [52]:
from gensim.models import Word2Vec
import multiprocessing

# See how many cpu core available — use them all for faster training
NUM_WORKERS = multiprocessing.cpu_count()
print(f'CPU cores available: {NUM_WORKERS}')
print('Training Word2Vec ...')

model = Word2Vec(
    sentences=playlists,
    vector_size=128,   # embedding dimensions
    window=15,        # context window
    negative=40,      # negative sampling
    min_count=5,      # ignore movies appearing in fewer than 5 playlists
    workers=NUM_WORKERS,
    epochs=20,
    seed=42
)

print(f'\n Model trained!')
print(f'Movies learned : {len(model.wv):,}')
print(f'Embedding size : {model.wv.vector_size}')


CPU cores available: 4
Training Word2Vec ...

 Model trained!
Movies learned : 13,663
Embedding size : 128


---
## 4. Movie Recommendation Function

In [85]:
def recommend_movies(movie_id, topn=5):
    """Recommend similar movies based on their embedding."""
    mid = str(movie_id)
    if mid not in model.wv:
        print(f'Movie {movie_id} not in model or too few ratings.')
        return None

    similar = model.wv.most_similar(positive=mid, topn=topn)

    rows = []
    for sid, score in similar:
        if sid in movie_meta.index:
            rows.append({
                'movieId': int(sid),
                'title': movie_meta.loc[sid, 'title'],
                'genres': movie_meta.loc[sid, 'genres'],
                'similarity': round(score, 3)
            })

    # Display query movie info
    if mid in movie_meta.index:
        q = movie_meta.loc[mid]
        print(f"{q['title']}")
        print(f"Genres: {q['genres']}")

    print(f'\n Top {topn} recommendations:')
    return pd.DataFrame(rows)[['movieId', 'title', 'genres', 'similarity']]


# Search helper — find movieId by title
def find_movie(query):
    """Find movieId by partial title match."""
    mask = movies['title'].str.contains(query, case=False, na=False)
    return movies[mask][['movieId', 'title', 'genres']].head(5)


find_movie('Toy Story')

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
3027,3114,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy
15401,78499,Toy Story 3 (2010),Adventure|Animation|Children|Comedy|Fantasy|IMAX
21981,106022,Toy Story of Terror (2013),Animation|Children|Comedy
24458,115875,Toy Story Toons: Hawaiian Vacation (2011),Adventure|Animation|Children|Comedy|Fantasy


In [86]:
recommend_movies(78499)

Toy Story 3 (2010)
Genres: Adventure|Animation|Children|Comedy|Fantasy|IMAX

 Top 5 recommendations:


,movieId,title,genres,similarity
0,79091,Despicable Me (2010),Animation|Children|Comedy|Crime,0.623
1,81591,Black Swan (2010),Drama|Thriller,0.615
2,72998,Avatar (2009),Action|Adventure|Sci-Fi|IMAX,0.604
3,72226,Fantastic Mr. Fox (2009),Adventure|Animation|Children|Comedy|Crime,0.583
4,81845,"King's Speech, The (2010)",Drama,0.582


In [87]:
find_movie('Shrek')

,movieId,title,genres
4211,4306,Shrek (2001),Adventure|Animation|Children|Comedy|Fantasy|Ro...
7761,8360,Shrek 2 (2004),Adventure|Animation|Children|Comedy|Musical|Ro...
11871,53121,Shrek the Third (2007),Adventure|Animation|Children|Comedy|Fantasy
13193,64249,Shrek the Halls (2007),Adventure|Animation|Comedy|Fantasy
15425,78637,Shrek Forever After (a.k.a. Shrek: The Final C...,Adventure|Animation|Children|Comedy|Fantasy|IMAX


In [84]:
recommend_movies(53121)

Shrek the Third (2007)
Genres: Adventure|Animation|Children|Comedy|Fantasy

 Top 5 recommendations:


,movieId,title,genres,similarity
0,52694,Mr. Bean's Holiday (2007),Comedy,0.863
1,52287,Meet the Robinsons (2007),Action|Adventure|Animation|Children|Comedy|Sci-Fi,0.831
2,52722,Spider-Man 3 (2007),Action|Adventure|Sci-Fi|Thriller|IMAX,0.830
3,53460,Surf's Up (2007),Animation|Children|Comedy,0.820
4,53993,Evan Almighty (2007),Comedy|Fantasy,0.787


In [66]:
find_movie('Harry Potter')

,movieId,title,genres
4800,4896,Harry Potter and the Sorcerer's Stone (a.k.a. ...,Adventure|Children|Fantasy
5717,5816,Harry Potter and the Chamber of Secrets (2002),Adventure|Fantasy
7769,8368,Harry Potter and the Prisoner of Azkaban (2004),Adventure|Fantasy|IMAX
10600,40815,Harry Potter and the Goblet of Fire (2005),Adventure|Fantasy|Thriller|IMAX
11974,54001,Harry Potter and the Order of the Phoenix (2007),Adventure|Drama|Fantasy|IMAX


In [ ]:
recommend_movies(54001)

Harry Potter and the Order of the Phoenix (2007)
Genres: Adventure|Drama|Fantasy|IMAX

 Top 5 recommendations:


,movieId,title,genres,similarity
0,56152,Enchanted (2007),Adventure|Animation|Children|Comedy|Fantasy|Mu...,0.584
1,53121,Shrek the Third (2007),Adventure|Animation|Children|Comedy|Fantasy,0.567
2,54259,Stardust (2007),Adventure|Comedy|Fantasy|Romance,0.564
3,52975,Hairspray (2007),Comedy|Drama|Musical,0.561
4,54272,"Simpsons Movie, The (2007)",Animation|Comedy,0.552
